## Momenti angolari reali in base di Wigner per B⁺ → K⁺π⁺π⁻ℓ⁻ℓ⁺

Questo notebook riparte dai risultati salvati da `AmpiezzaWigner.ipynb` e corregge la costruzione della base reale.

Dopo la serie di Clebsch-Gordan il modulo quadro è scritto come
$$|\mathcal{A}|^2=\sum_k A_k\,\mathcal{D}_k(\Omega),\qquad \mathcal{D}_k = d^{j_\ell}_{m_\ell,k_\ell}(\theta_\ell)\,D^{j_1}_{m_1,k_1}(\phi,\theta_1,0)\,D^{j_2}_{m_2,k_2}(\chi,\theta_2,0).$$
Per ogni prodotto $\mathcal{D}$ esiste il partner $\mathcal{D}'$ con tutti gli indici cambiati di segno, e vale $\mathcal{D}^*=p\,\mathcal{D}'$ con $p=(-1)^{\sum(m-k)}$.
Poiché $|\mathcal{A}|^2$ è reale, il coefficiente del partner è $A'=p\,A^*$, quindi
$$A\,\mathcal{D}+A'\,\mathcal{D}' = 2\,\mathrm{Re}(A\,\mathcal{D}) = \underbrace{2\,\mathrm{Re}A}_{\text{momento}}\;\mathrm{Re}\,\mathcal{D}\;+\;\underbrace{(-2\,\mathrm{Im}A)}_{\text{momento}}\;\mathrm{Im}\,\mathcal{D}.$$
Ogni coppia dà quindi **due** funzioni reali, $\mathrm{Re}\,\mathcal{D}$ e $\mathrm{Im}\,\mathcal{D}$, ciascuna con il suo momento reale.
Se $\mathcal{D}=\mathcal{D}'$ (tutti gli indici nulli) la funzione è già reale e il momento è $A$, che deve essere reale.

La versione precedente teneva solo $\mathrm{Re}\,\mathcal{D}$ con coefficiente $A+A'$: perdeva il termine in $\mathrm{Im}\,\mathcal{D}$ e, quando $p=-1$, produceva un coefficiente immaginario.

Le funzioni angolari restano scritte con le matrici di Wigner, senza espanderle in seni e coseni.
Gli altri problemi discussi (ampiezza non risonante, raggio di Blatt-Weisskopf, ecc.) **non** sono trattati qui.

### Setup

In [1]:
import os, time, random
import dill
import numpy as np
import sympy as sp
from sympy.physics.quantum.spin import Rotation as rot, WignerD
from sympy.core.function import AppliedUndef
from tqdm.auto import tqdm

sp.init_printing(use_unicode=True, wrap_line=True, num_columns=160)

# Cartella di lavoro: quella che contiene i .pkl prodotti da AmpiezzaWigner.ipynb
WORKDIR = os.getcwd()
OUTDIR = os.path.join(WORKDIR, "Moments_ReIm")
os.makedirs(OUTDIR, exist_ok=True)

phi, chi, theta1, theta2, thetal = sp.symbols('phi chi theta1 theta2 thetal', real=True)

### Caricamento dei risultati salvati
- `angular_basis.pkl`, `angular_trig_coefficients.pkl`: prodotti di tre Wigner D (complessi) e relativi coefficienti, cioè lo stato **prima** della costruzione della base reale.
- `WIGAtotsq_terms*.pkl`: il modulo quadro dell'ampiezza, usato solo per la verifica numerica finale.

In [2]:
t0 = time.time()
with open("angular_basis.pkl", "rb") as f:
    angular_basis = dill.load(f)
with open("angular_trig_coefficients.pkl", "rb") as f:
    angular_trig_coefficients = dill.load(f)

atotsq_file = sorted(f for f in os.listdir(WORKDIR) if f.startswith("WIGAtotsq_terms") and f.endswith(".pkl"))[-1]
with open(atotsq_file, "rb") as f:
    Atotsq = dill.load(f)["Atotsq"]

print(f"Prodotti complessi di Wigner D: {len(angular_basis)}")
print(f"Termini totali nei coefficienti: {sum(len(sp.Add.make_args(v)) for v in angular_trig_coefficients.values())}")
print(f"Termini nel modulo quadro ({atotsq_file}): {len(sp.Add.make_args(Atotsq))}")
print(f"Tempo di caricamento: {time.time()-t0:.1f} s")

Prodotti complessi di Wigner D: 515
Termini totali nei coefficienti: 216841
Termini nel modulo quadro (WIGAtotsq_terms33281.pkl): 33281
Tempo di caricamento: 25.5 s


### Valutatore numerico
Per i controlli serve valutare velocemente espressioni enormi. Si assegnano valori casuali a tutti i parametri dinamici:
complessi per $a_i$, $H$, $\bar H$, $G$, $R$; reali positivi per i fattori di Blatt-Weisskopf, $q$ e i raggi.
Il valutatore percorre l'albero dell'espressione invece di usare `subs`, che sarebbe troppo lento.

In [3]:
# Tutti i parametri dinamici presenti
dyn_atoms = set()
for v in angular_trig_coefficients.values():
    dyn_atoms |= v.atoms(AppliedUndef)
    dyn_atoms |= {s for s in v.free_symbols}

def is_real_param(a):
    if isinstance(a, AppliedUndef):
        return a.func.__name__ == 'BW'
    return bool(a.is_real)

def draw_values(seed):
    rng = random.Random(seed)
    vals = {}
    for a in dyn_atoms:
        if is_real_param(a):
            vals[a] = rng.uniform(0.5, 1.5)
        else:
            vals[a] = complex(rng.uniform(-1, 1), rng.uniform(-1, 1))
    return vals

_num_cache, _wig_cache = {}, {}
def fval(f, V, ang=None):
    # Valore numerico complesso di f, dati i parametri V e (opzionale) gli angoli ang
    if f.is_number:
        r = _num_cache.get(f)
        if r is None:
            r = _num_cache[f] = complex(sp.N(f))
        return r
    if f in V:
        return complex(V[f])
    if f.is_Mul:
        r = 1
        for g in f.args:
            r *= fval(g, V, ang)
        return r
    if f.is_Add:
        return sum(fval(g, V, ang) for g in f.args)
    if f.is_Pow:
        return fval(f.base, V, ang) ** int(f.exp)
    if isinstance(f, sp.conjugate):
        return fval(f.args[0], V, ang).conjugate()
    if isinstance(f, sp.re):
        return complex(fval(f.args[0], V, ang).real)
    if isinstance(f, sp.im):
        return complex(fval(f.args[0], V, ang).imag)
    if isinstance(f, sp.Abs):
        return complex(abs(fval(f.args[0], V, ang)))
    if isinstance(f, WignerD):
        key = (f, tuple(sorted(ang.items(), key=str)))
        r = _wig_cache.get(key)
        if r is None:
            r = _wig_cache[key] = complex(sp.N(f.doit().subs(ang)))
        return r
    raise TypeError(f"Oggetto non valutabile: {f} ({type(f)})")

V1, V2 = draw_values(1), draw_values(2)
t0 = time.time()
num1 = {k: fval(v, V1) for k, v in angular_trig_coefficients.items()}
num2 = {k: fval(v, V2) for k, v in angular_trig_coefficients.items()}
print(f"Valutazione numerica dei coefficienti: {time.time()-t0:.1f} s")

Valutazione numerica dei coefficienti: 3.7 s


### Accoppiamento $\mathcal{D}\leftrightarrow\mathcal{D}'$
Per ogni prodotto si costruisce il partner $\mathcal{D}'$ e la fase $p$.
Si verifica numericamente, per due scelte casuali dei parametri, che $A' = p\,A^*$: è una conseguenza della realtà di $|\mathcal{A}|^2$
e quindi un controllo di consistenza di tutto il calcolo precedente, non una scelta.

In [4]:
def partner(DDD):
    new, p = sp.S(1), 1
    for D in DDD.as_ordered_factors():
        J, M, K = D.args[:3]
        new *= rot.D(J, -M, -K, *D.args[3:])
        p *= (-1) ** int(M - K)
    return new, p

blocks = []          # ('self', DDD) oppure ('pair', DDD, p)
used = set()
max_dev = 0.0
for DDD in angular_basis:
    if DDD in used:
        continue
    P, p = partner(DDD)
    if P == DDD:
        assert p == 1
        used.add(DDD)
        blocks.append(('self', DDD))
        for num in (num1, num2):
            max_dev = max(max_dev, abs(num[DDD].imag) / max(abs(num[DDD]), 1e-300))
        continue
    if P not in angular_trig_coefficients:
        raise ValueError(f"Partner mancante per {DDD}")
    used |= {DDD, P}
    blocks.append(('pair', DDD, p))
    for num in (num1, num2):
        A, Ap = num[DDD], num[P]
        max_dev = max(max_dev, abs(Ap - p * A.conjugate()) / max(abs(A), 1e-300))

n_self = sum(b[0] == 'self' for b in blocks)
n_pair = sum(b[0] == 'pair' for b in blocks)
print(f"Prodotti già reali (D = D'): {n_self}")
print(f"Coppie (D, D'):              {n_pair}")
print(f"Massima deviazione relativa da A' = p A* (e da Im A = 0 per i prodotti reali): {max_dev:.2e}")
assert max_dev < 1e-10

Prodotti già reali (D = D'): 75
Coppie (D, D'):              220
Massima deviazione relativa da A' = p A* (e da Im A = 0 per i prodotti reali): 2.96e-14


### Momenti in forma esplicitamente reale
Ogni termine del coefficiente $A$ è scritto come $c\,r$, con $c$ costante numerica e $r$ prodotto di parametri.
Da $r$ si separano i fattori reali ($B_L$, $q$) e le coppie $z\,z^*\to|z|^2$; quello che resta, $\rho$, è un prodotto complesso
che viene ridotto a una forma canonica scelta tra $\rho$ e $\rho^*$.
Raccogliendo, $\mathrm{Re}A$ e $\mathrm{Im}A$ diventano somme di termini del tipo $\mathrm{Re}(\rho)$ e $\mathrm{Im}(\rho)$, come in Tabella II di Dey.

Il momento associato a $\mathrm{Re}\,\mathcal{D}$ è $2\,\mathrm{Re}A$, quello associato a $\mathrm{Im}\,\mathcal{D}$ è $-2\,\mathrm{Im}A$,
quello associato a un prodotto già reale è $\mathrm{Re}A$.

In [5]:
def is_real_factor(f):
    if f.is_number:
        return bool(f.is_real)
    return sp.conjugate(f) == f

def split_term(t):
    # t = c * real_part * rho, con c numero, real_part reale, rho complesso (o 1)
    c, real_part, rest = sp.S(1), [], []
    for f in sp.Mul.make_args(t):
        if f.is_number:
            c *= f
        elif is_real_factor(f):
            real_part.append(f)
        else:
            rest.append(f)
    # coppie z, conj(z) -> |z|^2
    rho = []
    pool = list(rest)
    while pool:
        f = pool.pop(0)
        fc = sp.conjugate(f)
        if fc in pool:
            pool.remove(fc)
            base = f.args[0] if isinstance(f, sp.conjugate) else f
            real_part.append(sp.Abs(base) ** 2)
        else:
            rho.append(f)
    return c, sp.Mul(*real_part), sp.Mul(*rho)

def real_imag_parts(A):
    # Restituisce (Re A, Im A) come espressioni esplicitamente reali
    acc = {}   # (real_part, rho_canonico) -> [X, Y]: A contiene X*rp*rho + Y*rp*conj(rho)
    for t in sp.Add.make_args(sp.expand(A)):
        c, rp, rho = split_term(t)
        rhoc = sp.conjugate(rho)
        if rho == rhoc:                      # termine reale
            key, slot = (rp, sp.S(1)), 0
            c = c * rho
        elif sp.default_sort_key(rho) <= sp.default_sort_key(rhoc):
            key, slot = (rp, rho), 0
        else:
            key, slot = (rp, rhoc), 1
        acc.setdefault(key, [sp.S(0), sp.S(0)])[slot] += c
    re_terms, im_terms = [], []
    for (rp, rho), (X, Y) in acc.items():
        if rho == 1:
            Xs = X
            re_terms.append(sp.re(Xs) * rp)
            im_terms.append(sp.im(Xs) * rp)
            continue
        # X rho + Y rho* : Re = Re((X+Y*) rho), Im = Im((X-Y*) rho)
        U = sp.expand(X + sp.conjugate(Y))
        W = sp.expand(X - sp.conjugate(Y))
        Rr, Ir = sp.re(rho, evaluate=False), sp.im(rho, evaluate=False)
        re_terms.append(rp * (sp.re(U) * Rr - sp.im(U) * Ir))
        im_terms.append(rp * (sp.re(W) * Ir + sp.im(W) * Rr))
    return sp.Add(*re_terms), sp.Add(*im_terms)

functions = []   # lista di dizionari: chiave = funzione angolare, momento = espressione reale
for b in tqdm(blocks, desc="Momenti reali"):
    DDD = b[1]
    ReA, ImA = real_imag_parts(angular_trig_coefficients[DDD])
    if b[0] == 'self':
        functions.append(dict(D=DDD, part='self', moment=ReA, num1=num1[DDD].real, num2=num2[DDD].real))
    else:
        functions.append(dict(D=DDD, part='re', moment=2 * ReA,
                              num1=2 * num1[DDD].real, num2=2 * num2[DDD].real))
        functions.append(dict(D=DDD, part='im', moment=-2 * ImA,
                              num1=-2 * num1[DDD].imag, num2=-2 * num2[DDD].imag))
print(f"Funzioni reali candidate: {len(functions)}")

Momenti reali:   0%|          | 0/295 [00:00<?, ?it/s]

Funzioni reali candidate: 515


### Controlli sui momenti
1. Ogni momento simbolico deve coincidere numericamente con $2\mathrm{Re}A$, $-2\mathrm{Im}A$ o $A$ calcolati direttamente.
2. Si scartano i momenti identicamente nulli (nulli in entrambe le scelte casuali dei parametri).

In [6]:
max_rel = 0.0
for F in tqdm(functions, desc="Controllo"):
    for V, ref in ((V1, F['num1']), (V2, F['num2'])):
        val = fval(F['moment'], V)
        assert abs(val.imag) < 1e-10 * max(1, abs(val)), "momento non reale!"
        max_rel = max(max_rel, abs(val.real - ref) / max(abs(ref), 1e-12))
print(f"Massimo scarto relativo tra momento simbolico e valore diretto: {max_rel:.2e}")
assert max_rel < 1e-8

scale = max(abs(F['num1']) for F in functions)
is_zero = [abs(F['num1']) < 1e-12 * scale and abs(F['num2']) < 1e-12 * scale for F in functions]
zero = [F for F, z in zip(functions, is_zero) if z]
functions = [F for F, z in zip(functions, is_zero) if not z]
print(f"Momenti identicamente nulli scartati: {len(zero)}")

Controllo:   0%|          | 0/515 [00:00<?, ?it/s]

Massimo scarto relativo tra momento simbolico e valore diretto: 2.14e-12
Momenti identicamente nulli scartati: 0


### Funzioni angolari
Le funzioni restano nella forma di Wigner: $\mathrm{Re}(\mathcal{D})$, $\mathrm{Im}(\mathcal{D})$, oppure $\mathcal{D}$ se già reale.
Si ordinano per rango dei tre fattori ($j_\ell$, $j_1$, $j_2$) e poi per indici.

In [7]:
def ranks(DDD):
    out = {}
    for D in DDD.as_ordered_factors():
        J, M, K = D.args[:3]
        ang = D.args[3:]
        g = 'l' if thetal in ang else ('1' if theta1 in ang else '2')
        out[g] = (int(J), int(M), int(K))
    return out

def sort_key(F):
    r = ranks(F['D'])
    return (r['l'][0], r['1'][0], r['2'][0], r['l'][1:], r['1'][1:], r['2'][1:], {'self': 0, 're': 1, 'im': 2}[F['part']])

def angular_function(F):
    r = ranks(F['D'])
    D = (rot.D(*r['l'], 0, thetal, 0) * rot.D(*r['1'], phi, theta1, 0) * rot.D(*r['2'], chi, theta2, 0))
    if F['part'] == 're':
        return sp.re(D, evaluate=False)
    if F['part'] == 'im':
        return sp.im(D, evaluate=False)
    return D

# Se tutti gli indici M sono nulli, D non dipende dagli azimut ed è reale anche se K != 0:
# in quel caso Im(D) è identicamente nulla come funzione e va tolta dalla base.
# Succede solo per prodotti con indice leptonico k != 0, generati dal termine non risonante.
null_fun = [F for F in functions if F['part'] == 'im' and all(v[1] == 0 for v in ranks(F['D']).values())]
print(f"Funzioni Im(D) identicamente nulle tolte dalla base: {len(null_fun)}")
functions = [F for F in functions if not (F['part'] == 'im' and all(v[1] == 0 for v in ranks(F['D']).values()))]

functions.sort(key=sort_key)
for i, F in enumerate(functions, 1):
    F['index'] = i
    F['r'] = ranks(F['D'])
    F['f'] = angular_function(F)
print(f"Numero di funzioni angolari reali con momento non nullo: {len(functions)}")
print(f"  di cui già reali:      {sum(F['part']=='self' for F in functions)}")
print(f"  di cui parte reale:    {sum(F['part']=='re' for F in functions)}")
print(f"  di cui parte immag.:   {sum(F['part']=='im' for F in functions)}")

Funzioni Im(D) identicamente nulle tolte dalla base: 7
Numero di funzioni angolari reali con momento non nullo: 508
  di cui già reali:      75
  di cui parte reale:    220
  di cui parte immag.:   213


### Verifica sul modulo quadro dell'ampiezza
Si confronta $\sum_i G_i\,f_i(\Omega)$ con il modulo quadro salvato `Atotsq`, in punti casuali degli angoli e dei parametri.
Se la ricostruzione è corretta i due valori coincidono.

In [8]:
rng = random.Random(7)
recon_err = 0.0
for V in (V1, V2):
    for _ in range(3):
        ang = {phi: rng.uniform(0, 2*np.pi), chi: rng.uniform(0, 2*np.pi),
               theta1: rng.uniform(0, np.pi), theta2: rng.uniform(0, np.pi), thetal: rng.uniform(0, np.pi)}
        direct = fval(Atotsq, V, ang)
        recon = sum(fval(F['moment'], V) * fval(F['f'], V, ang) for F in functions)
        print(f"|A|^2 diretto = {direct.real: .10e} (Im {direct.imag:.1e})   ricostruito = {recon.real: .10e}   scarto rel. = {abs(direct-recon)/abs(direct):.1e}")
        recon_err = max(recon_err, abs(direct - recon) / abs(direct))

|A|^2 diretto =  1.4055532306e+00 (Im -2.5e-17)   ricostruito =  1.4055532306e+00   scarto rel. = 7.6e-15


|A|^2 diretto =  1.3053757846e+00 (Im 1.4e-16)   ricostruito =  1.3053757846e+00   scarto rel. = 2.1e-14


|A|^2 diretto =  1.5863114438e+00 (Im -2.5e-17)   ricostruito =  1.5863114438e+00   scarto rel. = 1.2e-14


|A|^2 diretto =  2.9017637533e+00 (Im 3.9e-16)   ricostruito =  2.9017637533e+00   scarto rel. = 1.7e-14


|A|^2 diretto =  2.8579901236e+00 (Im 1.0e-16)   ricostruito =  2.8579901236e+00   scarto rel. = 1.2e-14


|A|^2 diretto =  3.4698711290e+00 (Im 1.6e-16)   ricostruito =  3.4698711290e+00   scarto rel. = 1.4e-14


### Separazione L/R e simmetria $\omega_i$
Nel limite di massa nulla $G_i = g_i^L + \omega_i\, g_i^R$ come in Dey (eq. 34) e nella tesi (eq. 6.26).
Si separano i termini con chiralità $\eta=-1$ e $\eta=+1$ leggendo il primo argomento di $H$, $\bar H$, $G$.
I termini senza ampiezze adroniche, cioè $|a_{NR}|^2$, sono comparsi una volta per ciascuna chiralità e vengono divisi a metà.
$\omega_i$ si ottiene confrontando **numericamente** $g^R_i$ con $g^L_i$ dopo la sostituzione $\eta=-1\to+1$.

In [9]:
HADR = {'H', 'Hbar', 'G'}

def term_eta(t):
    etas = {a.args[0] for a in t.atoms(AppliedUndef) if a.func.__name__ in HADR}
    if not etas:
        return None
    assert len(etas) == 1, t
    return int(list(etas)[0])

def flip_eta(expr):
    rep = {a: a.func(-a.args[0], *a.args[1:]) for a in expr.atoms(AppliedUndef) if a.func.__name__ in HADR}
    return expr.xreplace(rep)

for F in tqdm(functions, desc="L/R"):
    gL, gR = [], []
    for t in sp.Add.make_args(sp.expand(F['moment'])):
        e = term_eta(t)
        if e == -1:
            gL.append(t)
        elif e == 1:
            gR.append(t)
        else:
            gL.append(t / 2); gR.append(t / 2)
    F['gL'], F['gR'] = sp.Add(*gL), sp.Add(*gR)
    # Per il confronto servono valori per i parametri con eta=+1: si usano quelli già estratti
    w = None
    vals = []
    for V in (V1, V2):
        a = fval(flip_eta(F['gL']), V).real
        b = fval(F['gR'], V).real
        vals.append((a, b))
    if all(abs(b - a) < 1e-9 * max(abs(a), abs(b), 1e-30) for a, b in vals):
        w = 1
    elif all(abs(b + a) < 1e-9 * max(abs(a), abs(b), 1e-30) for a, b in vals):
        w = -1
    F['omega'] = w

from collections import Counter
print(Counter(F['omega'] for F in functions))

L/R:   0%|          | 0/508 [00:00<?, ?it/s]

Counter({1: 298, -1: 174, None: 36})


### Ortogonalità della base
Si calcola la matrice di Gram con il prodotto scalare dell'eq. 6.16 della tesi.
Ogni funzione è $P(\theta_\ell,\theta_1,\theta_2)$ per $\cos(m_1\phi+m_2\chi)$ o $\sin(m_1\phi+m_2\chi)$,
quindi l'integrale azimutale è analitico e quelli polari si fanno con quadratura di Gauss-Legendre in $\theta$.
Questo controllo **non** corregge nulla: serve solo a vedere quali funzioni non sono ortogonali.

In [10]:
xg, wg = np.polynomial.legendre.leggauss(80)
th = 0.5 * np.pi * (xg + 1); wth = 0.5 * np.pi * wg * np.sin(th)   # misura d(cos theta)

_dcache = {}
def dvals(j, m, k):
    key = (j, m, k)
    if key not in _dcache:
        b = sp.Symbol('b')
        fn = sp.lambdify(b, rot.d(j, m, k, b).doit(), 'numpy')
        _dcache[key] = np.broadcast_to(np.asarray(fn(th), dtype=float), th.shape)
    return _dcache[key]

def polar(F, G, g):
    rf, rg = F['r'][g], G['r'][g]
    return np.sum(wth * dvals(*rf) * dvals(*rg))

def azim(F, G):
    # (1/4pi^2) * integrale su phi, chi
    a = (F['r']['1'][1], F['r']['2'][1]); b = (G['r']['1'][1], G['r']['2'][1])
    tf = 'cos' if F['part'] in ('re', 'self') else 'sin'
    tg = 'cos' if G['part'] in ('re', 'self') else 'sin'
    sf = -1 if F['part'] == 'im' else 1          # Im(e^{-ix}) = -sin x
    sg = -1 if G['part'] == 'im' else 1
    na = (-a[0], -a[1])
    if tf != tg:
        return 0.0
    if a == (0, 0) and b == (0, 0):
        return 1.0 * sf * sg if tf == 'cos' else 0.0
    same, opp = (a == b), (na == b)
    val = 0.5 * same + (0.5 if tf == 'cos' else -0.5) * opp
    return sf * sg * val

n = len(functions)
Gram = np.zeros((n, n))
for i in range(n):
    for j in range(i, n):
        az = azim(functions[i], functions[j])
        if az == 0.0:
            continue
        v = az * polar(functions[i], functions[j], 'l') * polar(functions[i], functions[j], '1') * polar(functions[i], functions[j], '2') / 8
        Gram[i, j] = Gram[j, i] = v

norms = np.sqrt(np.diag(Gram))
Corr = Gram / np.outer(norms, norms)
off = [(i, j, Corr[i, j]) for i in range(n) for j in range(i+1, n) if abs(Corr[i, j]) > 1e-8]
print(f"Coppie non ortogonali: {len(off)}")
bad = sorted({functions[i]['index'] for i, j, _ in off} | {functions[j]['index'] for i, j, _ in off})
print(f"Funzioni coinvolte: {len(bad)}")
lep_k = [F['index'] for F in functions if ranks(F['D'])['l'][2] != 0]
print(f"Funzioni con indice leptonico k != 0: {len(lep_k)}")
print(f"Le funzioni non ortogonali hanno tutte k_l != 0 oppure sono accoppiate a una con k_l != 0: "
      f"{all(ranks(functions[i]['D'])['l'][2] != 0 or ranks(functions[j]['D'])['l'][2] != 0 for i, j, _ in off)}")

Coppie non ortogonali: 114
Funzioni coinvolte: 101
Funzioni con indice leptonico k != 0: 47
Le funzioni non ortogonali hanno tutte k_l != 0 oppure sono accoppiate a una con k_l != 0: True


### Funzioni indipendenti
Le funzioni angolari sono ortogonali, quindi come funzioni sono tutte linearmente indipendenti.
La ridondanza sta nei **momenti**: alcuni $G_k$ sono combinazioni lineari fisse di altri,
$G_k = \sum_j c_{kj}\, G_j$, per qualunque valore dei parametri dinamici ($a_i$, $H$, $\bar H$, $G$, $R$, $B_L$, $q$).
In quel caso il termine $G_k f_k$ non porta informazione nuova e si può riassorbire:
$$\frac{d\Gamma}{d\Omega}\propto\sum_{j\in\text{indip.}} G_j\, \tilde f_j,\qquad \tilde f_j = f_j + \sum_{k\in\text{dip.}} c_{kj}\, f_k .$$

**Metodo.** Si valutano tutti i momenti in molti punti casuali dello spazio dei parametri (più punti che funzioni) e si costruisce la matrice
$M_{ri} = G_i(\text{punto } r)$. Le colonne vengono esaminate in ordine, con Gram-Schmidt: una colonna è indipendente se non è
combinazione delle precedenti. L'ordine dà la priorità alle funzioni con indice leptonico $k=0$ e, a parità, a quelle con indice minore,
così l'insieme scelto è il più "semplice" possibile.
Per ogni momento dipendente si ricavano i coefficienti $c_{kj}$ con i minimi quadrati e si verificano su punti nuovi.

L'indipendenza è intesa per parametri dinamici generici e indipendenti: relazioni aggiuntive imposte dalla fisica
(per esempio i fattori di forma LEET) potrebbero ridurre ulteriormente il numero.
Le funzioni $\tilde f_j$ in generale non sono più ortogonali tra loro: lo si misura alla fine della cella con la matrice di Gram.

In [11]:
# --- 1. Compilazione dei coefficienti A in forma numerica vettoriale ---
atom_list = sorted(dyn_atoms, key=str)
aidx = {a: i for i, a in enumerate(atom_list)}

def compile_factor(f):
    # restituisce (indice atomo, coniugato?, potenza) oppure None se f è un numero
    p = 1
    if f.is_Pow:
        f, p = f.base, int(f.exp)
    conj = False
    if isinstance(f, sp.conjugate):
        f, conj = f.args[0], True
    return aidx[f], conj, p

def compile_coeff(A):
    terms = []
    for t in sp.Add.make_args(A):
        c, facs = 1.0 + 0j, []
        for f in sp.Mul.make_args(t):
            if f.is_number:
                c *= fval(f, {})          # numeri con cache
            else:
                facs.append(compile_factor(f))
        terms.append((c, facs))
    return terms

block_of = {}           # D -> coefficiente compilato (uno per prodotto complesso)
for F in functions:
    if F['D'] not in block_of:
        block_of[F['D']] = compile_coeff(angular_trig_coefficients[F['D']])

def sample_matrix(n_draws, seed):
    # Matrice M[r, i] = G_i nel punto r dello spazio dei parametri
    rng = np.random.default_rng(seed)
    Vm = np.empty((len(atom_list), n_draws), dtype=complex)
    for i, a in enumerate(atom_list):
        if is_real_param(a):
            Vm[i] = rng.uniform(0.5, 1.5, n_draws)
        else:
            Vm[i] = rng.uniform(-1, 1, n_draws) + 1j * rng.uniform(-1, 1, n_draws)
    Vc = Vm.conj()
    Aval = {}
    for D, terms in block_of.items():
        acc = np.zeros(n_draws, dtype=complex)
        for c, facs in terms:
            v = np.full(n_draws, c)
            for i, conj, p in facs:
                x = Vc[i] if conj else Vm[i]
                v = v * (x if p == 1 else x ** p)
            acc += v
        Aval[D] = acc
    M = np.empty((n_draws, len(functions)))
    for col, F in enumerate(functions):
        A = Aval[F['D']]
        M[:, col] = {'self': A.real, 're': 2 * A.real, 'im': -2 * A.imag}[F['part']]
    return M

t0 = time.time()
n_draws = 2 * len(functions) + 100
M = sample_matrix(n_draws, seed=11)
print(f"Matrice dei momenti: {M.shape}, calcolata in {time.time()-t0:.1f} s")

# Controllo di coerenza con i valori calcolati prima (stesso valutatore su punti diversi: si confronta solo la scala)
assert np.all(np.isfinite(M))

# --- 2. Selezione greedy con Gram-Schmidt modificato ---
order = sorted(range(len(functions)), key=lambda i: (functions[i]['r']['l'][2] != 0, functions[i]['index']))
Mn = M / np.linalg.norm(M, axis=0)
TOL = 1e-8
Q = np.zeros((n_draws, 0))
indep, dep = [], []
for i in order:
    v = Mn[:, i].copy()
    for _ in range(2):                      # doppia ortogonalizzazione per stabilità
        v -= Q @ (Q.T @ v)
    nv = np.linalg.norm(v)
    if nv > TOL:
        indep.append(i)
        Q = np.column_stack([Q, v / nv])
    else:
        dep.append(i)

sv = np.linalg.svd(Mn, compute_uv=False)
print(f"Rango numerico (greedy): {len(indep)}")
print(f"Valori singolari attorno al salto: {sv[len(indep)-2:len(indep)+2]}")

# --- 3. Relazioni lineari per i momenti dipendenti, verificate su punti nuovi ---
M_test = sample_matrix(60, seed=12)
indep_sorted = sorted(indep, key=lambda i: functions[i]['index'])
Mi, Mi_test = M[:, indep_sorted], M_test[:, indep_sorted]
relations = {}
max_res = 0.0
for k in dep:
    c, *_ = np.linalg.lstsq(Mi, M[:, k], rcond=None)
    c[np.abs(c) < 1e-9 * np.abs(c).max()] = 0.0
    res = np.linalg.norm(Mi_test @ c - M_test[:, k]) / np.linalg.norm(M_test[:, k])
    max_res = max(max_res, res)
    relations[k] = {indep_sorted[j]: c[j] for j in np.nonzero(c)[0]}
print(f"Massimo residuo relativo delle relazioni su punti nuovi: {max_res:.1e}")
assert max_res < 1e-6

from fractions import Fraction
def exact_guess(x):
    # i coefficienti vengono da prodotti di Clebsch-Gordan: si cerca la forma ±sqrt(razionale)
    # si prende il denominatore più piccolo compatibile con la precisione numerica
    for lim in (10, 100, 1000):
        f = Fraction(x * x).limit_denominator(lim)
        g = sp.sign(x) * sp.sqrt(sp.Rational(f.numerator, f.denominator))
        if abs(float(g) - x) < 1e-10 * max(1, abs(x)):
            return g
    return sp.Float(x, 12)

for F in functions:
    F['independent'] = None
    F['relation'] = None
for i in indep:
    functions[i]['independent'] = True
for k, rel in relations.items():
    functions[k]['independent'] = False
    functions[k]['relation'] = {functions[j]['index']: exact_guess(c) for j, c in rel.items()}

# --- 4. Funzioni efficaci f~_j = f_j + sum_k c_kj f_k e loro ortogonalità ---
Cmat = np.zeros((len(functions), len(indep_sorted)))      # f~ = F^T C
for col, j in enumerate(indep_sorted):
    Cmat[j, col] = 1.0
for k, rel in relations.items():
    for j, c in rel.items():
        Cmat[k, indep_sorted.index(j)] += c
for col, j in enumerate(indep_sorted):
    F = functions[j]
    F['f_eff'] = F['f'] + sum(sp.Float(Cmat[k, col], 12) * functions[k]['f']
                              for k in range(len(functions)) if k != j and Cmat[k, col] != 0)
G_eff = Cmat.T @ Gram @ Cmat
d = np.sqrt(np.clip(np.diag(G_eff), 1e-300, None))
Corr_eff = G_eff / np.outer(d, d)
np.fill_diagonal(Corr_eff, 0)

phys_ind = [functions[i] for i in indep if functions[i]['r']['l'][2] == 0]
print()
print(f"Funzioni totali: {len(functions)}")
print(f"  momenti indipendenti: {len(indep)}  (di cui con k_l = 0: {len(phys_ind)})")
print(f"  momenti dipendenti:   {len(dep)}")
print(f"  simmetria tra gli indipendenti: +1 -> {sum(functions[i]['omega']==1 for i in indep)}, "
      f"-1 -> {sum(functions[i]['omega']==-1 for i in indep)}, non definita -> {sum(functions[i]['omega'] is None for i in indep)}")
n_exact = sum(not isinstance(c, sp.Float) for F in functions if F['relation'] for c in F['relation'].values())
n_coef = sum(len(F['relation']) for F in functions if F['relation'])
print(f"Coefficienti c_kj riconosciuti come ±sqrt(razionale): {n_exact}/{n_coef}")
up = np.abs(np.triu(Corr_eff))
print(f"Funzioni efficaci f~ non ortogonali tra loro: {int(np.sum(up > 1e-8))} coppie su {len(indep)*(len(indep)-1)//2}")
print(f"  con |correlazione| > 0.5: {int(np.sum(up > 0.5))},  > 0.9: {int(np.sum(up > 0.9))},  massima: {up.max():.6f}")
ii, jj = np.unravel_index(np.argmax(up), up.shape)
print(f"  coppia più correlata: f~_{functions[indep_sorted[ii]]['index']} e f~_{functions[indep_sorted[jj]]['index']}")

print("\nEsempi di relazioni:")
for k in sorted(relations, key=lambda k: functions[k]['index'])[:8]:
    rel = functions[k]['relation']
    rhs = " + ".join(f"({v})·G_{j}" for j, v in rel.items())
    print(f"  G_{functions[k]['index']} = {rhs}")

Matrice dei momenti: (1116, 508), calcolata in 1.9 s
Rango numerico (greedy): 381
Valori singolari attorno al salto: [1.29593878e-02 3.51753285e-03 1.21310349e-15 1.15090077e-15]


Massimo residuo relativo delle relazioni su punti nuovi: 5.5e-13

Funzioni totali: 508
  momenti indipendenti: 381  (di cui con k_l = 0: 341)
  momenti dipendenti:   127
  simmetria tra gli indipendenti: +1 -> 217, -1 -> 133, non definita -> 31
Coefficienti c_kj riconosciuti come ±sqrt(razionale): 295/336
Funzioni efficaci f~ non ortogonali tra loro: 431 coppie su 72390
  con |correlazione| > 0.5: 154,  > 0.9: 27,  massima: 1.000000
  coppia più correlata: f~_131 e f~_137

Esempi di relazioni:
  G_31 = (10/7)·G_5 + (-15/7)·G_16
  G_42 = (-20*sqrt(3)/3)·G_4 + (22*sqrt(6)/3)·G_14 + (14*sqrt(2))·G_27 + (13.9664546519)·G_29
  G_44 = (4)·G_16
  G_54 = (48/7)·G_4 + (-5*sqrt(2))·G_14 + (-3*sqrt(6))·G_27 + (-6.91160258818)·G_29
  G_55 = (sqrt(10))·G_14 + (3*sqrt(30)/5)·G_27
  G_57 = (-8*sqrt(15)/15)·G_29
  G_58 = (-8*sqrt(15)/15)·G_30
  G_59 = (18/7)·G_5 + (-20/7)·G_16


### Riepilogo e salvataggio
- `REIM_moments.pkl`: tutte le funzioni, con i campi `independent` e `relation` (per i dipendenti: $G_k=\sum_j c_{kj}G_j$, chiavi = indici $j$).
- `REIM_moments_independent.pkl`: solo le funzioni indipendenti, con la funzione efficace `f_eff` $=\tilde f_j$.
- Cartella `Moments_ReIm`: un file `D_<indice>_omega_<piu|meno|misto>.txt` per ogni funzione **indipendente**
  (funzione angolare, funzione efficace, $\omega_i$, momento $g_i^L$) e il file `relazioni.txt` con le relazioni dei momenti dipendenti.

In [12]:
records = []
for F in functions:
    records.append(dict(index=F['index'], f=F['f'], D=F['D'], part=F['part'],
                        omega=F['omega'], gL=F['gL'], gR=F['gR'], moment=F['moment'],
                        independent=F['independent'], relation=F['relation'],
                        f_eff=F.get('f_eff')))
with open("REIM_moments.pkl", "wb") as fh:
    dill.dump(records, fh)
records_ind = [R for R in records if R['independent']]
with open("REIM_moments_independent.pkl", "wb") as fh:
    dill.dump(records_ind, fh)

for old in os.listdir(OUTDIR):
    if old.endswith(".txt"):
        os.remove(os.path.join(OUTDIR, old))
for R in records_ind:
    tag = {1: 'piu', -1: 'meno', None: 'misto'}[R['omega']]
    with open(os.path.join(OUTDIR, f"D_{R['index']:03d}_omega_{tag}.txt"), "w", encoding="utf-8") as fh:
        fh.write(f"D_{R['index']} = {R['f']}\n")
        fh.write(f"f_eff_{R['index']} = {R['f_eff']}\n")
        fh.write(f"omega = {R['omega']}\n\n")
        fh.write("g^L =\n")
        fh.write(str(R['gL']) + "\n")
with open(os.path.join(OUTDIR, "relazioni.txt"), "w", encoding="utf-8") as fh:
    for R in records:
        if R['independent'] is False:
            rhs = " + ".join(f"({v})*G_{j}" for j, v in R['relation'].items())
            fh.write(f"G_{R['index']} = {rhs}    [D_{R['index']} = {R['f']}]\n")

print(f"Funzioni angolari totali: {len(records)}")
print(f"Funzioni indipendenti salvate: {len(records_ind)}")
for w, name in ((1, '+(L->R)'), (-1, '-(L->R)'), (None, 'non definita')):
    print(f"  simmetria {name}: {sum(R['omega'] == w for R in records_ind)}")

aNR = sp.Symbol('a_NR', complex=True)
phys = [F for F in functions if F['r']['l'][2] == 0]
art = [F for F in functions if F['r']['l'][2] != 0]
print()
print(f"Funzioni con indice leptonico k = 0: {len(phys)} (indipendenti: {sum(bool(F['independent']) for F in phys)})")
print(f"Funzioni con indice leptonico k != 0 (tutte da interferenza con a_NR: "
      f"{all(aNR in F['moment'].free_symbols for F in art)}): {len(art)} (indipendenti: {sum(bool(F['independent']) for F in art)})")

Funzioni angolari totali: 508
Funzioni indipendenti salvate: 381
  simmetria +(L->R): 217
  simmetria -(L->R): 133
  simmetria non definita: 31

Funzioni con indice leptonico k = 0: 461 (indipendenti: 341)
Funzioni con indice leptonico k != 0 (tutte da interferenza con a_NR: True): 47 (indipendenti: 40)


### Esempi: le prime funzioni e i rispettivi momenti $g^L$

In [13]:
for R in records[:3]:
    print(f"D_{R['index']}  (omega = {R['omega']})")
    sp.pprint(R['f'])
    print(f"numero di termini in g^L: {len(sp.Add.make_args(R['gL']))}")
    print()
# una funzione Re e la sua Im
R_re = next(R for R in records if R['part'] == 're')
R_im = next(R for R in records if R['part'] == 'im' and R['D'] == R_re['D'])
for R in (R_re, R_im):
    print(f"D_{R['index']}  (omega = {R['omega']})")
    sp.pprint(R['f'])
    sp.pprint(R['gL'])
    print()

D_1  (omega = 1)
 0            0            0          
d   (thetal)⋅D   (χ,θ₂,0)⋅D   (φ,θ₁,0)
 0,0          0,0          0,0        
numero di termini in g^L: 418

D_2  (omega = 1)
 0            0            1          
d   (thetal)⋅D   (φ,θ₁,0)⋅D   (χ,θ₂,0)
 0,0          0,0          0,0        
numero di termini in g^L: 109

D_3  (omega = 1)
 0            0            2          
d   (thetal)⋅D   (φ,θ₁,0)⋅D   (χ,θ₂,0)
 0,0          0,0          0,0        
numero di termini in g^L: 406

D_8  (omega = 1)
  ⎛ 0            1            1          ⎞
re⎜d   (thetal)⋅D   (φ,θ₁,0)⋅D   (χ,θ₂,0)⎟
  ⎝ 0,0          0,1          1,0        ⎠


      2                                                              ⎛                                                     __ _________________________ ______ ↪
  √6⋅q ⋅BW(1, r₂, K*(1410))⋅BW(1, r₂, K*(892))⋅BW(1, r₃, K1(1650))⋅re⎝a₂₃⋅H̅(-1, 1, -1, 0, K*(1410))⋅R(K*(1410))⋅R(K*(892))⋅a₃⋅H(-1, 1, -1, 0, K1(1650))⋅R(K1(1 ↪
- ──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────── ↪
                                                                                    2                                                                          ↪
                                                                                12⋅π                                                                           ↪

↪ _____⎞       2                                                              ⎛                                                    __ ________________________ ↪
↪ 650))⎠   √6⋅q ⋅BW(1, r₂, K*(14

    2                                                              ⎛                                                     __ _________________________ ________ ↪
√6⋅q ⋅BW(1, r₂, K*(1410))⋅BW(1, r₂, K*(892))⋅BW(1, r₃, K1(1650))⋅im⎝a₂₃⋅H̅(-1, 1, -1, 0, K*(1410))⋅R(K*(1410))⋅R(K*(892))⋅a₃⋅H(-1, 1, -1, 0, K1(1650))⋅R(K1(165 ↪
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────── ↪
                                                                                  2                                                                            ↪
                                                                              12⋅π                                                                             ↪

↪ ___⎞       2                                                              ⎛                                                    __ ________________________ _ ↪
↪ 0))⎠   √6⋅q ⋅BW(1, r₂, K*(1410

### Riepilogo dettagliato
Tutti i numeri sono letti dalle variabili calcolate sopra, non scritti a mano.
L'ordine segue la priorità dell'analisi dei momenti: prima la **base ortogonale** di funzioni angolari, poi il numero di **momenti indipendenti**.

In [14]:
from collections import Counter

def line(s=""):
    print(s)

phys_idx = [n for n, F in enumerate(functions) if F['r']['l'][2] == 0]
art_idx  = [n for n, F in enumerate(functions) if F['r']['l'][2] != 0]
phys_set = set(phys_idx)
up_all = np.abs(np.triu(Corr, 1))
nonorth_phys = sum(1 for i, j, _ in off if i in phys_set and j in phys_set)
nonorth_mixed = sum(1 for i, j, _ in off if (i in phys_set) != (j in phys_set))
nonorth_art = sum(1 for i, j, _ in off if i not in phys_set and j not in phys_set)
aNR = sp.Symbol('a_NR', complex=True)

line("=" * 78)
line("1. PUNTO DI PARTENZA (risultati salvati da AmpiezzaWigner.ipynb)")
line("=" * 78)
line(f"   Termini nel modulo quadro |A|^2:                    {len(sp.Add.make_args(Atotsq))}")
line(f"   Prodotti complessi di tre Wigner D dopo la serie CG: {len(angular_basis)}")
line(f"     - prodotti già reali (D = D'):                    {n_self}")
line(f"     - coppie (D, D') con D* = p D':                   {n_pair}")
line(f"   Verifica A' = p A* sulle coppie (scarto rel. max):  {max_dev:.1e}")
line(f"   Vecchia base (solo Re D, coefficiente A + A'):      {n_self + n_pair} funzioni, informazione incompleta")

line()
line("=" * 78)
line("2. BASE ORTOGONALE DI FUNZIONI ANGOLARI REALI (forma di Wigner)")
line("=" * 78)
line(f"   Funzioni candidate: {n_self} (D reale) + {n_pair} (Re D) + {n_pair} (Im D) = {n_self + 2*n_pair}")
line(f"   Im D identicamente nulle (tutti gli M = 0), tolte:  {len(null_fun)}")
line(f"   Momenti identicamente nulli, tolti:                 {len(zero)}")
line(f"   FUNZIONI NELLA BASE:                                {len(functions)}")
cnt = Counter(F['part'] for F in functions)
line(f"     - D già reali: {cnt['self']},  Re D: {cnt['re']},  Im D: {cnt['im']}")
line()
line(f"   Indice leptonico k = 0 (d^j_(m,0)(theta_l)):        {len(phys_idx)}")
line(f"     coppie non ortogonali tra queste:                 {nonorth_phys}   -> base ortogonale")
w = Counter(functions[n]['omega'] for n in phys_idx)
line(f"     simmetria L->R: +1 -> {w[1]}, -1 -> {w[-1]}, non definita -> {w[None]}")
line(f"   Indice leptonico k != 0:                            {len(art_idx)}")
line(f"     tutte con a_NR nel momento:                       {all(aNR in functions[n]['moment'].free_symbols for n in art_idx)}")
w = Counter(functions[n]['omega'] for n in art_idx)
line(f"     simmetria L->R: +1 -> {w[1]}, -1 -> {w[-1]}, non definita -> {w[None]}")
line(f"     coppie non ortogonali tra loro:                   {nonorth_art}")
line(f"     coppie non ortogonali con funzioni k = 0:         {nonorth_mixed}")
line(f"   Totale coppie non ortogonali nella base:            {len(off)} (tutte coinvolgono k != 0)")
line()
line(f"   Momenti simbolici vs valori diretti (scarto max):   {max_rel:.1e}")
line(f"   Ricostruzione di |A|^2 in punti casuali (scarto max): {recon_err:.1e}")

line()
line("=" * 78)
line("3. MOMENTI INDIPENDENTI")
line("=" * 78)
ind_phys = [i for i in indep if i in phys_set]
ind_art = [i for i in indep if i not in phys_set]
line(f"   Rango della matrice dei momenti ({M.shape[0]} punti x {M.shape[1]} funzioni): {len(indep)}")
line(f"   Salto nei valori singolari: {sv[len(indep)-1]:.1e} -> {sv[len(indep)]:.1e}")
line(f"   Momenti indipendenti tra le {len(phys_idx)} funzioni con k = 0: {len(ind_phys)}")
line(f"     (le funzioni k = 0 sono esaminate per prime: {len(ind_phys)} è il rango dei soli momenti k = 0)")
line(f"   Momenti indipendenti aggiunti dalle funzioni k != 0:  {len(ind_art)}")
line(f"   Momenti dipendenti (relazioni G_k = sum c_kj G_j):    {len(dep)}")
line(f"     di cui tra funzioni k = 0:                        {sum(1 for k in dep if k in phys_set)}")
line(f"   Residuo massimo delle relazioni su punti nuovi:     {max_res:.1e}")
line(f"   Coefficienti c_kj della forma ±sqrt(razionale):     {n_exact}/{n_coef} (gli altri sono somme di radici, lasciati numerici)")
w = Counter(functions[i]['omega'] for i in indep)
line(f"   Simmetria dei momenti indipendenti: +1 -> {w[1]}, -1 -> {w[-1]}, non definita -> {w[None]}")
line()
line("   Se si riassorbono i dipendenti nelle funzioni efficaci f~_j, l'ortogonalità si perde:")
line(f"     coppie f~ non ortogonali: {int(np.sum(up > 1e-8))},  |corr| > 0.9: {int(np.sum(up > 0.9))},  massima: {up.max():.6f}")

line()
line("=" * 78)
line("4. FILE SALVATI")
line("=" * 78)
line(f"   REIM_moments.pkl              tutte le {len(records)} funzioni (campi independent, relation)")
line(f"   REIM_moments_independent.pkl  le {len(records_ind)} funzioni con momento indipendente (con f_eff)")
line(f"   {os.path.basename(OUTDIR)}/                  {len(records_ind)} file .txt + relazioni.txt")

1. PUNTO DI PARTENZA (risultati salvati da AmpiezzaWigner.ipynb)
   Termini nel modulo quadro |A|^2:                    33281
   Prodotti complessi di tre Wigner D dopo la serie CG: 515
     - prodotti già reali (D = D'):                    75
     - coppie (D, D') con D* = p D':                   220
   Verifica A' = p A* sulle coppie (scarto rel. max):  3.0e-14
   Vecchia base (solo Re D, coefficiente A + A'):      295 funzioni, informazione incompleta

2. BASE ORTOGONALE DI FUNZIONI ANGOLARI REALI (forma di Wigner)
   Funzioni candidate: 75 (D reale) + 220 (Re D) + 220 (Im D) = 515
   Im D identicamente nulle (tutti gli M = 0), tolte:  7
   Momenti identicamente nulli, tolti:                 0
   FUNZIONI NELLA BASE:                                508
     - D già reali: 75,  Re D: 220,  Im D: 213

   Indice leptonico k = 0 (d^j_(m,0)(theta_l)):        461
     coppie non ortogonali tra queste:                 0   -> base ortogonale
     simmetria L->R: +1 -> 296, -1 -> 165, non def